In [13]:
%reload_ext autoreload
%autoreload 2

In [14]:
# --------------------- Import VQNiche ---------------------
from vqniche.utils.parse_test_configs import *
from vqniche.initializers.initialize import *
from vqniche.utils.type_conversions import *
from vqniche.plotting import *

In [15]:
# --------------------- Import Libraries ---------------------
import os
import copy
import sys
import yaml
import pickle
from pathlib import Path
from dataclasses import dataclass

import scanpy as sc
import anndata as ad
import squidpy as sq

import numpy as np
import networkx as nx
import scipy.sparse as sp
from scipy.stats import pearsonr

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import pytorch_lightning as pl
import torch_geometric.transforms as T
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj
from torch_geometric.loader import DataLoader as BatchBuilder

# --------------------- Display Settings ---------------------
# display setting all rows and columns
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [16]:
save_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/paper/tables")
save_dir.mkdir(parents=True, exist_ok=True)

In [17]:
def collect_wandb_run_dirs(
        sweep_dir,
        type_of_run: Literal['run', 'offline-run']='run'
    ):
    wandb_run_dirs = []
    for d in os.listdir(sweep_dir / "wandb"):
        if type_of_run in d:
            # print(d)
            wandb_run_dirs.append(sweep_dir / "wandb" / d)
    return wandb_run_dirs

In [18]:
def build_sweep_test_results_df(
        wandb_run_dirs,
        split: Literal['1-test-patch', '3-test-patch']='1-test-patch',
        conditioning=True
    ):
    wandb_run_dir_dfs = []
    for wandb_run_dir in wandb_run_dirs:
        if isinstance(wandb_run_dir, str):
            cfg = f"{wandb_run_dir}/files/user_specified_config.yaml"
            test_res_file = f"{wandb_run_dir}/files/results/test_metrics.csv"
        else:
            cfg = wandb_run_dir/"files"/"user_specified_config.yaml"
            test_res_file = wandb_run_dir/"files"/"results"/"test_metrics.csv"
        with open(cfg, 'r') as f:
            cfg = yaml.safe_load(f)
        test_results_df = pd.read_csv(test_res_file)
        
        test_results_df['dataset'] = cfg['dataset']['dataset_name']
        test_results_df['seed'] = cfg['experiment']['seed']
        if conditioning:
            test_results_df['model'] = cfg['model']['model_name']
        else:
            test_results_df['model'] = f"{cfg['model']['model_name']} w/o C"
        test_results_df['split'] = split

        wandb_run_dir_dfs.append(test_results_df)
        
    sweep_results = pd.concat(
        objs=wandb_run_dir_dfs,
        ignore_index=True
    )

    return sweep_results

In [19]:
def build_clean_sweep_results_df(
        df: pd.DataFrame,
        metrics: List[str] = [
            "pearson_1hop_nbr",
            "pearson_gene_wise_1hop_nbr",
            "mmd_1hop_nbr",
            "mmd_pca_1hop_nbr",
        ]
    ):
    # --- Step 1: groupby and transpose ---
    df_long = df.melt(
        id_vars=["dataset", "seed", "model", "split", "epoch"], 
        var_name="metric", 
        value_name="value"
    )
    
    # --- Step 2: strip mode from metric name ---
    df_long["mode"] = df_long["metric"].str.extract(r"^(train|val|test)")
    df_long["metric"] = df_long["metric"].str.replace(r"^(train|val|test)_", "", regex=True)
    df_long = df_long[df_long['metric'].isin(metrics)]

    # --- Step 3: pretty names for metric ---
    metric_rename = {
        "pearson_1hop_nbr": "PC1",
        "pearson_gene_wise_1hop_nbr": "PG1",
        "pearson_cell_wise": "PC0",
        "mmd_1hop_nbr": "MMDC1",
        "mmd_pca_1hop_nbr": "MMD-PCA-C-1hop",
    }
    df_long["metric"] = df_long["metric"].map(metric_rename)

    # --- Step 4: pretty names for model ---
    model_rename = {
        "VQNiche": "SQUINT",
        "VQNiche w/o C": "SQUINT w/o C",
        "WFM": "WFM",
    }
    df_long["model"] = df_long["model"].map(model_rename)
    
    # --- Step 5: pretty names for dataset ---
    dataset_rename = {
        "xhs1000-39b_1p": "Skin (Human)",
        "mmb0-4b_1p": "Brain (Mouse)",
        "xhk1020-CV1-CV2-5b_1p": "Kidney (Human)",
    }
    df_long["dataset"] = df_long["dataset"].map(dataset_rename)
    
    # --- Step 6: pretty names for split ---
    split_rename = {
        "1-test-patch": "1 Patch",
        "3-test-patch": "3 Patches",
    }
    df_long["split"] = df_long["split"].map(split_rename)
    
    # --- Step 7: round to 4 decimals ---
    df_long['value'] = df_long['value'].round(4)
    
    df_long = df_long.drop_duplicates()
    
    return df_long

In [20]:
df_datasets = []
metrics = [
    "pearson_1hop_nbr",
    "pearson_cell_wise",
    # "pearson_gene_wise_1hop_nbr",
    "mmd_1hop_nbr",
    # "mmd_pca_1hop_nbr"
]

# Dataset: xhs1000-39b_1p-oriented-3

## 3 Test Regions

### No Conditioning

In [21]:
sweep_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhs1000-39b_1p/sweep/VQNiche/batch=[2, 11, 12]/spatial_n_neighs_8/seed/20250918-122511")
wandb_run_dirs = collect_wandb_run_dirs(sweep_dir)
sweep_results = build_sweep_test_results_df(
    wandb_run_dirs,
    split='3-test-patch',
    conditioning=False,
)
display(sweep_results.head(3))

df_long = build_clean_sweep_results_df(
                sweep_results,
                metrics,
            )
display(df_long.head(3))

df_datasets.append(df_long)

,test_codebook_utilization,test_pearson_cell_wise,test_pearson_1hop_nbr,test_pearson_gene_wise,test_pearson_gene_wise_1hop_nbr,test_mmd_1hop_nbr,test_mmd_pca_1hop_nbr,epoch,dataset,seed,model,split
0,0.151,0.393367,0.727116,0.118851,0.282960,0.091764,0.003596,4,xhs1000-39b_1p,1,VQNiche w/o C,3-test-patch
1,0.105,0.400144,0.725740,0.112954,0.269126,0.096759,0.003591,4,xhs1000-39b_1p,0,VQNiche w/o C,3-test-patch
2,0.128,0.395007,0.727992,0.124209,0.299491,0.095290,0.003609,4,xhs1000-39b_1p,2,VQNiche w/o C,3-test-patch


,dataset,seed,model,split,epoch,metric,value,mode
4,Skin (Human),1,SQUINT w/o C,3 Patches,4,PC0,0.3934,test
5,Skin (Human),0,SQUINT w/o C,3 Patches,4,PC0,0.4001,test
6,Skin (Human),2,SQUINT w/o C,3 Patches,4,PC0,0.3950,test


### Conditioning on RBF Distances + Batch-ID

In [22]:
sweep_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhs1000-39b_1p/sweep/VQNiche/batch=[2, 11, 12]/spatial_n_neighs_8/seed/20250918-121510")
wandb_run_dirs = collect_wandb_run_dirs(sweep_dir)
sweep_results = build_sweep_test_results_df(
    wandb_run_dirs,
    split='3-test-patch',
    conditioning=True,
)
display(sweep_results.head(3))

df_long = build_clean_sweep_results_df(
                sweep_results,
                metrics,
            )
display(df_long.head(3))

df_datasets.append(df_long)

,test_codebook_utilization,test_pearson_cell_wise,test_pearson_1hop_nbr,test_pearson_gene_wise,test_pearson_gene_wise_1hop_nbr,test_mmd_1hop_nbr,test_mmd_pca_1hop_nbr,epoch,dataset,seed,model,split
0,0.1494,0.438878,0.767513,0.127245,0.301352,0.078045,0.003585,4,xhs1000-39b_1p,0,VQNiche,3-test-patch
1,0.1794,0.441650,0.774775,0.127762,0.292029,0.079204,0.003595,4,xhs1000-39b_1p,1,VQNiche,3-test-patch
2,0.1332,0.442483,0.778006,0.130563,0.295166,0.097059,0.003611,4,xhs1000-39b_1p,2,VQNiche,3-test-patch


,dataset,seed,model,split,epoch,metric,value,mode
4,Skin (Human),0,SQUINT,3 Patches,4,PC0,0.4389,test
5,Skin (Human),1,SQUINT,3 Patches,4,PC0,0.4416,test
6,Skin (Human),2,SQUINT,3 Patches,4,PC0,0.4425,test


# Dataset: mmb04-4b_1p

## 3 Test Regions

### No Conditioning

In [23]:
sweep_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/mmb0-4b_1p/sweep/VQNiche/batch=[0, 1, 2, 3]/spatial_n_neighs_8/seed/20250918-120551")
wandb_run_dirs = collect_wandb_run_dirs(sweep_dir)
sweep_results = build_sweep_test_results_df(
    wandb_run_dirs,
    split='3-test-patch',
    conditioning=False,
)
display(sweep_results.head(3))

df_long = build_clean_sweep_results_df(
                sweep_results,
                metrics,
            )
display(df_long)

df_datasets.append(df_long)

,test_codebook_utilization,test_pearson_cell_wise,test_pearson_1hop_nbr,test_pearson_gene_wise,test_pearson_gene_wise_1hop_nbr,test_mmd_1hop_nbr,test_mmd_pca_1hop_nbr,epoch,dataset,seed,model,split
0,0.0930,0.644390,0.896107,0.120460,0.289691,0.048073,0.00246,4,mmb0-4b_1p,0,VQNiche w/o C,3-test-patch
1,0.0814,0.641789,0.877828,0.117395,0.283326,0.044902,0.00246,4,mmb0-4b_1p,2,VQNiche w/o C,3-test-patch
2,0.0814,0.641789,0.877828,0.117395,0.283326,0.044902,0.00246,4,mmb0-4b_1p,2,VQNiche w/o C,3-test-patch


,dataset,seed,model,split,epoch,metric,value,mode
4,Brain (Mouse),0,SQUINT w/o C,3 Patches,4,PC0,0.6444,test
5,Brain (Mouse),2,SQUINT w/o C,3 Patches,4,PC0,0.6418,test
7,Brain (Mouse),1,SQUINT w/o C,3 Patches,4,PC0,0.6275,test
8,Brain (Mouse),0,SQUINT w/o C,3 Patches,4,PC1,0.8961,test
9,Brain (Mouse),2,SQUINT w/o C,3 Patches,4,PC1,0.8778,test
11,Brain (Mouse),1,SQUINT w/o C,3 Patches,4,PC1,0.8913,test
20,Brain (Mouse),0,SQUINT w/o C,3 Patches,4,MMDC1,0.0481,test
21,Brain (Mouse),2,SQUINT w/o C,3 Patches,4,MMDC1,0.0449,test
23,Brain (Mouse),1,SQUINT w/o C,3 Patches,4,MMDC1,0.0410,test


### Conditioning with RBF Distances + Batch-ID

In [24]:
sweep_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/mmb0-4b_1p/sweep/VQNiche/batch=[0, 1, 2, 3]/spatial_n_neighs_8/seed/20250918-120525")
wandb_run_dirs = collect_wandb_run_dirs(sweep_dir)
sweep_results = build_sweep_test_results_df(
    wandb_run_dirs,
    split='3-test-patch',
    conditioning=True,
)
display(sweep_results.head(3))

df_long = build_clean_sweep_results_df(
                sweep_results,
                metrics,
            )
display(df_long)

df_datasets.append(df_long)

,test_codebook_utilization,test_pearson_cell_wise,test_pearson_1hop_nbr,test_pearson_gene_wise,test_pearson_gene_wise_1hop_nbr,test_mmd_1hop_nbr,test_mmd_pca_1hop_nbr,epoch,dataset,seed,model,split
0,0.0976,0.665215,0.906643,0.127004,0.309582,0.037895,0.00246,4,mmb0-4b_1p,0,VQNiche,3-test-patch
1,0.1256,0.664743,0.920757,0.126219,0.293255,0.039721,0.00246,4,mmb0-4b_1p,1,VQNiche,3-test-patch
2,0.1098,0.665078,0.920793,0.126838,0.298615,0.040766,0.00246,4,mmb0-4b_1p,2,VQNiche,3-test-patch


,dataset,seed,model,split,epoch,metric,value,mode
4,Brain (Mouse),0,SQUINT,3 Patches,4,PC0,0.6652,test
5,Brain (Mouse),1,SQUINT,3 Patches,4,PC0,0.6647,test
6,Brain (Mouse),2,SQUINT,3 Patches,4,PC0,0.6651,test
8,Brain (Mouse),0,SQUINT,3 Patches,4,PC1,0.9066,test
9,Brain (Mouse),1,SQUINT,3 Patches,4,PC1,0.9208,test
10,Brain (Mouse),2,SQUINT,3 Patches,4,PC1,0.9208,test
20,Brain (Mouse),0,SQUINT,3 Patches,4,MMDC1,0.0379,test
21,Brain (Mouse),1,SQUINT,3 Patches,4,MMDC1,0.0397,test
22,Brain (Mouse),2,SQUINT,3 Patches,4,MMDC1,0.0408,test


# Dataset: xhk1020-CV1-CV2-5b_1p

## 3 Test Regions

### No Conditioning

In [31]:
sweep_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhk1020-CV1-CV2-5b_1p/sweep/VQNiche/batch=[0, 1, 2, 3, 4]/spatial_n_neighs_8/seed/20250918-142428")
wandb_run_dirs = collect_wandb_run_dirs(sweep_dir)
sweep_results = build_sweep_test_results_df(
    wandb_run_dirs,
    split='3-test-patch',
    conditioning=False,
)
display(sweep_results.head(3))

df_long = build_clean_sweep_results_df(
                sweep_results,
                metrics,
            )
display(df_long)

df_datasets.append(df_long)

,test_codebook_utilization,test_pearson_cell_wise,test_pearson_1hop_nbr,test_pearson_gene_wise,test_pearson_gene_wise_1hop_nbr,test_mmd_1hop_nbr,test_mmd_pca_1hop_nbr,epoch,dataset,seed,model,split
0,0.0154,0.413844,0.809109,0.040570,0.296446,0.026088,0.000850,1,xhk1020-CV1-CV2-5b_1p,2,VQNiche w/o C,3-test-patch
1,0.0194,0.433439,0.834501,0.044074,0.314543,0.025507,0.000849,1,xhk1020-CV1-CV2-5b_1p,0,VQNiche w/o C,3-test-patch
2,0.0154,0.413844,0.809109,0.040570,0.296446,0.026088,0.000850,1,xhk1020-CV1-CV2-5b_1p,2,VQNiche w/o C,3-test-patch


,dataset,seed,model,split,epoch,metric,value,mode
4,Kidney (Human),2,SQUINT w/o C,3 Patches,1,PC0,0.4138,test
5,Kidney (Human),0,SQUINT w/o C,3 Patches,1,PC0,0.4334,test
7,Kidney (Human),1,SQUINT w/o C,3 Patches,1,PC0,0.4655,test
8,Kidney (Human),2,SQUINT w/o C,3 Patches,1,PC1,0.8091,test
9,Kidney (Human),0,SQUINT w/o C,3 Patches,1,PC1,0.8345,test
11,Kidney (Human),1,SQUINT w/o C,3 Patches,1,PC1,0.8374,test
20,Kidney (Human),2,SQUINT w/o C,3 Patches,1,MMDC1,0.0261,test
21,Kidney (Human),0,SQUINT w/o C,3 Patches,1,MMDC1,0.0255,test
23,Kidney (Human),1,SQUINT w/o C,3 Patches,1,MMDC1,0.0247,test


### Conditioning with RBF Distances + Batch-ID

In [26]:
sweep_dir = Path("/lustre/scratch126/cellgen/lotfollahi/am84/VQNiche/logs/xhk1020-CV1-CV2-5b_1p/sweep/VQNiche/batch=[0, 1, 2, 3, 4]/spatial_n_neighs_8/seed/20250918-133207")
wandb_run_dirs = collect_wandb_run_dirs(sweep_dir)
sweep_results = build_sweep_test_results_df(
    wandb_run_dirs,
    split='3-test-patch',
    conditioning=True,
)
display(sweep_results.head(3))

df_long = build_clean_sweep_results_df(
                sweep_results,
                metrics,
            )
display(df_long)

df_datasets.append(df_long)

,test_codebook_utilization,test_pearson_cell_wise,test_pearson_1hop_nbr,test_pearson_gene_wise,test_pearson_gene_wise_1hop_nbr,test_mmd_1hop_nbr,test_mmd_pca_1hop_nbr,epoch,dataset,seed,model,split
0,0.024,0.508650,0.870036,0.058456,0.365237,0.018434,0.000848,1,xhk1020-CV1-CV2-5b_1p,0,VQNiche,3-test-patch
1,0.024,0.501942,0.867359,0.050387,0.366540,0.021097,0.000855,1,xhk1020-CV1-CV2-5b_1p,1,VQNiche,3-test-patch
2,0.025,0.497488,0.868739,0.054344,0.371600,0.018810,0.000851,1,xhk1020-CV1-CV2-5b_1p,2,VQNiche,3-test-patch


,dataset,seed,model,split,epoch,metric,value,mode
4,Kidney (Human),0,SQUINT,3 Patches,1,PC0,0.5086,test
5,Kidney (Human),1,SQUINT,3 Patches,1,PC0,0.5019,test
6,Kidney (Human),2,SQUINT,3 Patches,1,PC0,0.4975,test
8,Kidney (Human),0,SQUINT,3 Patches,1,PC1,0.8700,test
9,Kidney (Human),1,SQUINT,3 Patches,1,PC1,0.8674,test
10,Kidney (Human),2,SQUINT,3 Patches,1,PC1,0.8687,test
20,Kidney (Human),0,SQUINT,3 Patches,1,MMDC1,0.0184,test
21,Kidney (Human),1,SQUINT,3 Patches,1,MMDC1,0.0211,test
22,Kidney (Human),2,SQUINT,3 Patches,1,MMDC1,0.0188,test


# Final Table

In [32]:
df_avg = pd.concat(
            objs=df_datasets,
            ignore_index=True
        )
df_avg = df_avg.groupby(
        ['dataset', 'split', 'model', 'metric']
        )['value'].mean().reset_index()
df_avg = df_avg.round(4)
display(df_avg)

,dataset,split,model,metric,value
0,Brain (Mouse),3 Patches,SQUINT,MMDC1,0.0395
1,Brain (Mouse),3 Patches,SQUINT,PC0,0.6650
2,Brain (Mouse),3 Patches,SQUINT,PC1,0.9161
3,Brain (Mouse),3 Patches,SQUINT w/o C,MMDC1,0.0447
4,Brain (Mouse),3 Patches,SQUINT w/o C,PC0,0.6379
5,Brain (Mouse),3 Patches,SQUINT w/o C,PC1,0.8884
6,Kidney (Human),3 Patches,SQUINT,MMDC1,0.0194
7,Kidney (Human),3 Patches,SQUINT,PC0,0.5027
8,Kidney (Human),3 Patches,SQUINT,PC1,0.8687
9,Kidney (Human),3 Patches,SQUINT w/o C,MMDC1,0.0254


In [33]:
df_pivot = df_avg.pivot(index='model', columns=['dataset','metric'], values='value')
model_order = ['WFM',  'SQUINT w/o C', 'SQUINT']
df_pivot = df_pivot.reindex(model_order).reset_index()
df_pivot = df_pivot.fillna('--')
for i in range(df_pivot.shape[1]-1):
    df_pivot.insert(i*2+1,f'amps-{i}','&')
i += 1
df_pivot.insert(i*2+1,f'newline-{i}','\\\\')
display(df_pivot)

dataset,model,amps-0,Brain (Mouse),amps-1,Brain (Mouse),amps-2,Brain (Mouse),amps-3,Kidney (Human),amps-4,Kidney (Human),amps-5,Kidney (Human),amps-6,Skin (Human),amps-7,Skin (Human),amps-8,Skin (Human),newline-9
metric,,,MMDC1,,PC0,,PC1,,MMDC1,,PC0,,PC1,,MMDC1,,PC0,,PC1,
0,WFM,&,--,&,--,&,--,&,--,&,--,&,--,&,--,&,--,&,--,\\
1,SQUINT w/o C,&,0.0447,&,0.6379,&,0.8884,&,0.0254,&,0.4376,&,0.827,&,0.0946,&,0.3962,&,0.7269,\\
2,SQUINT,&,0.0395,&,0.665,&,0.9161,&,0.0194,&,0.5027,&,0.8687,&,0.0848,&,0.441,&,0.7734,\\


In [34]:
top_str = """\\begin{table*}[t]\n
\\small
\\captionsetup[sub]{skip=0pt} \n
\\centering  \n
\\setlength\\tabcolsep{3pt} \n
\\caption{2D Imputation Results.}
\label{tab:2D_imputation_results} \n
"""

header_str = """\\begin{tabular}{lcccccccc} \n
\\toprule \n
\\multirow{2}[2]{*}{\\textbf{Model}} & \\multicolumn{4}{c}{\\textbf{Brain (Mouse)}} & \\multicolumn{4}{c}{\\textbf{Skin (Human)}} \\\\ \n
\\cmidrule(lr){2-5} \\cmidrule(lr){6-9} \n
& MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC1 $\\uparrow$ & PG1 $\\uparrow$ & MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC1 $\\uparrow$ & PG1 $\\uparrow$ \\\\ \n
\\midrule \n
"""

header_str = """\\begin{tabular}{lcccccccc} \n
\\toprule \n
\\multirow{2}[2]{*}{\\textbf{Model}} & \\multicolumn{4}{c}{\\textbf{Brain (Mouse)}} & \\multicolumn{4}{c}{\\textbf{Skin (Human)}} \\\\ \n
\\cmidrule(lr){2-5} \\cmidrule(lr){6-9} \n
& MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ & MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ \\\\ \n
\\midrule \n
"""
   # WFM &      0.0542 &      0.1314 &     -- &      -- &     0.0887 &      0.2026 &      -- &      -- \\

header_str = """\\begin{tabular}{lcccccccccccc} \n
\\toprule \n
\\multirow{2}[2]{*}{\\textbf{Model}} & \\multicolumn{4}{c}{\\textbf{Brain (Mouse)}} & \\multicolumn{4}{c}{\\textbf{Skin (Human)}} & \\multicolumn{4}{c}{\\textbf{Skin (Kidney)}} \\\\ \n
\\cmidrule(lr){2-5} \\cmidrule(lr){6-9} \\cmidrule(lr){10-13} \n
& MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ & MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ & MMDPCA1 $\\downarrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ \\\\ \n
\\midrule \n
"""

   # WFM &      0.0542 &      0.1314 &     -- &      -- &     0.0887 &      0.2026 &      -- &      --    &      0.0039 &      0.1118 &     -- &      -- \\

header_str = """\\begin{tabular}{lccccccccc} \n
\\toprule \n
\\multirow{2}[2]{*}{\\textbf{Model}} & \\multicolumn{3}{c}{\\textbf{Brain (Mouse)}} & \\multicolumn{3}{c}{\\textbf{Kidney (Human)}} & \\multicolumn{3}{c}{\\textbf{Skin (Human)}}  \\\\ \n
\\cmidrule(lr){2-4} \\cmidrule(lr){5-7} \\cmidrule(lr){8-10} \n
& MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ & MMDC1 $\\downarrow$ & PC0 $\\uparrow$ & PC1 $\\uparrow$ \\\\ \n
\\midrule \n
"""

   # WFM &      0.1314 &     -- &      -- &  0.1118 &     -- &      -- &  0.2026 &      -- &      --         \\

data_str = df_pivot.to_string(header=False, index=False)

footer_str =  "\n\\bottomrule \n \\end{tabular} \n"

bottom_str = '\\end{table*}'


fname = save_dir / "2D_imputation_results.tex"
with open(fname, 'w') as fp:
    fp.write(top_str)
    fp.write(header_str)
    fp.write(data_str)
    fp.write(footer_str)
    fp.write(bottom_str)